In [2]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

In [3]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [5]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

response.output_text

'Yes—probably, depending on whether the course is still open and what your situation is.\n\nIf you want, I can help you figure it out. Typically you’d need to check:\n- whether enrollment is still open,\n- if there are any prerequisites,\n- whether there’s a waiting list,\n- and if you’ve missed the start date but can still catch up.\n\nIf you meant a specific course, send me the course name or link and I can help you draft a message asking to join.'

In [4]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [5]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [8]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"Can I join the course late? discovery join course enrollment late registration"}', call_id='call_Rdp369WTbQrdPtNuZmdF0dno', name='search', type='function_call', id='fc_0194936e9dc10ecd006a4c0651cdc08192a8c708e67c9d233e', namespace=None, status='completed')]

In [9]:
import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

In [12]:
print(result_json)

[
  {
    "id": "74eb249bbf",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "I just discovered the course. Can I still join?",
    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\u2019re still accepting submissions."
  },
  {
    "id": "977bf7786c",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?",
    "answer": "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."
  },
  {
    "id": "69d122f12e",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "Certificate: Can I follow the course in a self-paced

In [13]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [14]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"Can I join the course late? discovery join course enrollment late registration"}', call_id='call_Rdp369WTbQrdPtNuZmdF0dno', name='search', type='function_call', id='fc_0194936e9dc10ecd006a4c0651cdc08192a8c708e67c9d233e', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_Rdp369WTbQrdPtNuZmdF0dno',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "977bf7786c",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Course: I have registered for the LLM Zoomcamp. When can I ex

In [15]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text

'Yes — you can still join the course.\n\nIf you want a certificate, make sure to submit your project while submissions are still being accepted.'

In [16]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(774, 32)

In [18]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(774, 32)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.0001353


In [14]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

question = "I just discovered the course. Can I join it?"


In [15]:
messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

In [16]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

In [24]:
import json

def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [ ]:
messages.extend(response.output)

for item in response.output:
    print(item)

    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
    elif item.type == "message":
        print('ASSISTANT:')
        print(item.content[0].text)


ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join enrollment late registration FAQ"}', call_id='call_MRGY7kqz7g909w3j06FzSB3t', name='search', type='function_call', id='fc_0fbded849cc883ea006a4c504fe52c81a0be5f3e1c9f7912b1', namespace=None, status='completed')
ResponseFunctionToolCall(arguments='{"query":"course discovered join it can I enroll FAQ"}', call_id='call_8S4kiUBlOj1H2Nw6SZTv1fak', name='search', type='function_call', id='fc_0fbded849cc883ea006a4c504fe54481a0a7638a55f47c10b2', namespace=None, status='completed')
ResponseFunctionToolCall(arguments='{"query":"late join course enrollment FAQ discovered the course"}', call_id='call_2nmvKV5OI2UnuHmOda01H3KP', name='search', type='function_call', id='fc_0fbded849cc883ea006a4c504fe54c81a0afff654b08b504ef', namespace=None, status='completed')


In [32]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question},
    ]

    it = 1

    while True:
        print(f"Iteration {it}")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool],
        )

        messages.extend(response.output)


        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it += 1
        if not has_function_calls:
            break

    return last_answer

In [30]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results
and then perform more searches

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

question = "I just discovered the course. Can I join it?"

In [33]:
agent_loop(instructions, question)

Iteration 1
function_call: search {"query":"join the course enrollment discover course can I join after start FAQ"}
Iteration 2
function_call: search {"query":"registered accepted start learning submitting homework while form is open no need confirmation email register interest live cohort certificate FAQ"}
Iteration 3
ASSISTANT:
Yes — you can join even if you just discovered the course.

A few useful notes:
- You can start learning right away; the materials are available.
- If you want a certificate, you need to complete the course with the live cohort and submit your project while submissions are still open.
- You do **not** need a confirmation email to begin; registration is mainly to gauge interest.

If you want, I can also help you with how to start the course, where to find the materials, or how the homework/certificate process works.


'Yes — you can join even if you just discovered the course.\n\nA few useful notes:\n- You can start learning right away; the materials are available.\n- If you want a certificate, you need to complete the course with the live cohort and submit your project while submissions are still open.\n- You do **not** need a confirmation email to begin; registration is mainly to gauge interest.\n\nIf you want, I can also help you with how to start the course, where to find the materials, or how the homework/certificate process works.'

In [34]:
agent_loop(instructions, "How do I run Olama locally?")

Iteration 1
function_call: search {"query":"Olama locally run install local Ollama FAQ"}
Iteration 2
function_call: search {"query":"Ollama serve localhost 11434 run llama3 connection refused restart serve FAQ"}
Iteration 3
ASSISTANT:
To run **Ollama locally**:

1. **Install Ollama**
   - Go to: https://ollama.com/download
   - macOS: install the `.pkg`
   - Windows: install the `.msi`
   - Linux:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. **Start a local model**
   ```bash
   ollama run llama3
   ```
   This downloads the model, starts it locally, and opens a chat interface.

3. **Check that the local server is running**
   ```bash
   curl http://localhost:11434
   ```
   You should get a response showing available models.

4. **If you get a connection refused error**
   Restart the Ollama server:
   ```bash
   nohup ollama serve > nohup.out 2>&1 &
   ```

5. **Optional: use it from Python**
   ```bash
   pip install ollama
   ```
   Example:
   ```py

'To run **Ollama locally**:\n\n1. **Install Ollama**\n   - Go to: https://ollama.com/download\n   - macOS: install the `.pkg`\n   - Windows: install the `.msi`\n   - Linux:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. **Start a local model**\n   ```bash\n   ollama run llama3\n   ```\n   This downloads the model, starts it locally, and opens a chat interface.\n\n3. **Check that the local server is running**\n   ```bash\n   curl http://localhost:11434\n   ```\n   You should get a response showing available models.\n\n4. **If you get a connection refused error**\n   Restart the Ollama server:\n   ```bash\n   nohup ollama serve > nohup.out 2>&1 &\n   ```\n\n5. **Optional: use it from Python**\n   ```bash\n   pip install ollama\n   ```\n   Example:\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": "Hello!"}]\n   )\n\n   print(response[\'message\'][\'content\'])\n   ```\n\

In [15]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results
and then perform more searches

The question has to be about the course or its logistics, offtopic questions
shouldn't be answered.
If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

question = "I just discovered the course. Can I join it?"

In [14]:
agent_loop(instructions, "what's queen's gambit?")

NameError: name 'agent_loop' is not defined

In [1]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [6]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [7]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [8]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [9]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [12]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [16]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


-> Response received


-> Response received


In [17]:
result.cost

CostInfo(input_cost=Decimal('0.00236775'), output_cost=Decimal('0.0014445'), total_cost=Decimal('0.00381225'))

In [18]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results\nand then perform more searches\n\nThe question has to be about the course or its logistics, offtopic questions\nshouldn't be answered.\nIf the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the\nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None),
 EasyInputMessage(content='How do I run Olama locally?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"Olama locally run local course FAQ"}', call_id='call_kseYE1mXeDJge

In [19]:
result2 = runner.loop(
    prompt="How do I run a different model?",
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received


-> Response received


In [ ]:
runner.run()